In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import librosa
import librosa.display
import matplotlib.pyplot as plt
from tqdm import tqdm
from google.colab import files

DATA_DIR = '/content/drive/MyDrive/data'
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
VALID_DIR = os.path.join(DATA_DIR, 'valid')
TEST_DIR = os.path.join(DATA_DIR, 'test')

print("Libraries imported and Drive mounted successfully!")

In [ ]:
def extract_mel_spectrogram(file_path, max_pad_len=174):
    try:
        y, sr = librosa.load(file_path, sr=22050, duration=3.0)

        spectrogram = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        spectrogram = librosa.power_to_db(spectrogram, ref=np.max)

        if spectrogram.shape[1] < max_pad_len:
            pad_width = max_pad_len - spectrogram.shape[1]
            spectrogram = np.pad(spectrogram, pad_width=((0, 0), (0, pad_width)), mode='constant')
        else:
            spectrogram = spectrogram[:, :max_pad_len]

        return spectrogram
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

print("Preprocessing pipeline defined.")

In [ ]:
def load_dataset_from_folders(base_dir):
    X, y = [], []
    classes = ['female', 'male']

    for class_idx, class_name in enumerate(classes):
        folder_path = os.path.join(base_dir, class_name)
        if not os.path.exists(folder_path):
            continue
        print(f"Processing folder: {folder_path}")
        for file_name in tqdm(os.listdir(folder_path)):
            if file_name.endswith('.wav'):
                file_path = os.path.join(folder_path, file_name)
                features = extract_mel_spectrogram(file_path)
                if features is not None:
                    X.append(features)
                    y.append(class_idx)

    return np.array(X), np.array(y)

print("--- Loading Training Data ---")
X_train, y_train = load_dataset_from_folders(TRAIN_DIR)

print("--- Loading Validation Data ---")
X_valid, y_valid = load_dataset_from_folders(VALID_DIR)

X_train = X_train[..., np.newaxis]
X_valid = X_valid[..., np.newaxis]

print(f"\nData successfully loaded!")
print(f"Train shape: {X_train.shape}, Validation shape: {X_valid.shape}")

In [ ]:
model = models.Sequential([
    layers.Input(shape=(X_train.shape[1], X_train.shape[2], 1)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.2),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.2),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

EPOCHS = 15
BATCH_SIZE = 32

print("\n--- Starting Training ---")
history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_valid, y_valid)
)

weights_filename = 'voice_gender_model.weights.h5'
drive_save_path = os.path.join(DATA_DIR, weights_filename)

model.save_weights(weights_filename)
model.save_weights(drive_save_path)
print(f"\nWeights saved successfully to Drive at: {drive_save_path}")

print("Triggering automatic file download to your computer...")
files.download(weights_filename)

In [ ]:
import os
import numpy as np
import librosa
from tensorflow.keras import models, layers
from google.colab import output
from IPython.display import HTML, display
import soundfile as sf
import base64
import time

model = models.Sequential([
    layers.Input(shape=(128, 174, 1)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.2),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.2),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

weights_path = 'voice_gender_model.weights.h5'

if os.path.exists(weights_path):
    model.load_weights(weights_path)
    print("Model weights loaded successfully!")
else:
    DATA_DIR = '/content/drive/MyDrive/data'
    drive_path = os.path.join(DATA_DIR, weights_path)
    if os.path.exists(drive_path):
        model.load_weights(drive_path)
        print("Model weights loaded successfully from Google Drive!")
    else:
        print("Error: Could not find weights file.")

In [ ]:
def predict_gender(file_path, max_pad_len=174):
    try:
        y, sr = librosa.load(file_path, sr=22050, duration=3.0)
        spectrogram = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        spectrogram = librosa.power_to_db(spectrogram, ref=np.max)

        if spectrogram.shape[1] < max_pad_len:
            pad_width = max_pad_len - spectrogram.shape[1]
            spectrogram = np.pad(spectrogram, pad_width=((0, 0), (0, pad_width)), mode='constant')
        else:
            spectrogram = spectrogram[:, :max_pad_len]

        features = spectrogram[np.newaxis, ..., np.newaxis]
        prediction = model.predict(features, verbose=0)[0][0]

        print("\n=================================")
        if prediction > 0.5:
            confidence = prediction * 100
            print(f" RESULT: MALE Voice Detected")
            print(f" Confidence: {confidence:.2f}%")
        else:
            confidence = (1 - prediction) * 100
            print(f" RESULT: FEMALE Voice Detected")
            print(f" Confidence: {confidence:.2f}%")
        print("=================================\n")

    except Exception as e:
        print(f"Prediction Error: {e}")

In [ ]:
AUDIO_JS = """
<script>
var my_recorder;
var my_stream;

async function startRecording() {
  my_stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  var options = { mimeType : 'audio/webm' };
  my_recorder = new MediaRecorder(my_stream, options);
  var chunks = [];

  my_recorder.ondataavailable = e => chunks.push(e.data);
  my_recorder.onstop = async () => {
    var blob = new Blob(chunks, { type: 'audio/webm' });
    var reader = new FileReader();
    reader.readAsDataURL(blob);
    reader.onloadend = () => {
      var base64data = reader.result.split(',')[1];
      google.colab.kernel.invokeFunction('notebook.SaveAudio', [base64data], {});
    };
  };
  my_recorder.start();
}

function stopRecording() {
  my_recorder.stop();
  my_stream.getTracks().forEach(track => track.stop());
}
</script>
"""

def save_audio(base64data):
    try:
        decoded_data = base64.b64decode(base64data)
        with open('live_input.webm', 'wb') as f:
            f.write(decoded_data)

        data, sr = librosa.load('live_input.webm', sr=22050)
        sf.write('live_input.wav', data, sr)
        print("\nAudio recorded successfully! Processing prediction...")

        predict_gender('live_input.wav')
    except Exception as e:
        print(f"Error saving/processing audio: {e}")

output.register_callback('notebook.SaveAudio', save_audio)

def record_live_voice():
    display(HTML(AUDIO_JS))
    print("Recording started! Speak into your microphone now...")
    output.eval_js('startRecording()')
    time.sleep(3.5)
    print("Recording stopped. Analyzing...")
    output.eval_js('stopRecording()')

record_live_voice()